# 02 — Analyse SAE Features

Inspect the trained Sparse Autoencoder: activation statistics, dead features, and visual feature galleries.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

import sys; sys.path.insert(0, '..')

from src.models.sae import SparseAutoencoder
from src.naming.feature_namer import get_top_images, rank_features_by_variance

SAE_PATH   = Path('../models/sae_best.pt')
EMB_PATH   = Path('../data/processed/plantvillage_embeddings.npy')
PATHS_JSON = Path('../data/processed/plantvillage_image_paths.json')
NAMES_JSON = Path('../models/feature_names.json')

## Load model and compute activations

In [ ]:
embeddings  = np.load(EMB_PATH).astype(np.float32)
image_paths = json.loads(PATHS_JSON.read_text())

state    = torch.load(SAE_PATH, map_location='cpu')
input_dim  = embeddings.shape[1]
hidden_dim = state['encoder.weight'].shape[0]
sae = SparseAutoencoder(input_dim=input_dim, hidden_dim=hidden_dim)
sae.load_state_dict(state)
sae.eval()

with torch.no_grad():
    activations = sae.encode(torch.from_numpy(embeddings)).numpy()

print(f'Activations shape : {activations.shape}')
print(f'Sparsity (frac 0) : {(activations == 0).mean():.4f}')

## Dead features

In [ ]:
ever_active = (activations > 0).any(axis=0)
dead_count  = (~ever_active).sum()
print(f'Dead features : {dead_count} / {hidden_dim} ({100*dead_count/hidden_dim:.1f}%)')

## Activation distribution

In [ ]:
mean_acts = activations.mean(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(mean_acts, bins=100, log=True)
axes[0].set_title('Mean activation per feature')
axes[0].set_xlabel('Mean activation')
axes[0].set_ylabel('Count (log)')

axes[1].hist(activations[activations > 0].ravel(), bins=100, log=True)
axes[1].set_title('Distribution of non-zero activations')
axes[1].set_xlabel('Activation value')
plt.tight_layout()
plt.show()

## Top-K image gallery for a feature

In [ ]:
ranked = rank_features_by_variance(activations)
FEATURE_ID = ranked[0]  # Most variable feature
K = 8

fi = get_top_images(activations, image_paths, FEATURE_ID, k=K)

fig, axes = plt.subplots(2, K, figsize=(2 * K, 5))
for i, (path, val) in enumerate(zip(fi.top_paths, fi.top_activations)):
    img = Image.open(path).convert('RGB').resize((112, 112))
    axes[0, i].imshow(img)
    axes[0, i].set_title(f'{val:.2f}', fontsize=8)
    axes[0, i].axis('off')

for i, (path, val) in enumerate(zip(fi.bottom_paths, fi.bottom_activations)):
    img = Image.open(path).convert('RGB').resize((112, 112))
    axes[1, i].imshow(img)
    axes[1, i].set_title(f'{val:.2f}', fontsize=8)
    axes[1, i].axis('off')

fig.text(0.01, 0.75, 'HIGH', va='center', fontsize=10, fontweight='bold')
fig.text(0.01, 0.25, 'LOW',  va='center', fontsize=10, fontweight='bold')
plt.suptitle(f'Feature {FEATURE_ID}', fontsize=12)
plt.tight_layout()
plt.show()

## Named features (if available)

In [ ]:
if NAMES_JSON.exists():
    names = json.loads(NAMES_JSON.read_text())
    print('Named features:')
    for fid, name in sorted(names.items(), key=lambda x: int(x[0])):
        act_mean = activations[:, int(fid)].mean()
        print(f'  [{fid:>5}] {name:40s}  mean_act={act_mean:.4f}')
else:
    print('No feature names found — run scripts/name_features.py first.')